In [ ]:
from pathlib import Path

# Basic YOLOv8 training script (Ultralytics)
# - Update DATA_YAML to your dataset yaml path
# - Update RUNS_DIR if you want a custom output folder

from ultralytics import YOLO

# --- User config (edit these) ---
DATA_YAML = r"D:\First try\justtry\data.yaml"   # TODO: set your dataset yaml
RUNS_DIR = r"E:\\path\\to\\runs"               # TODO: optional; set to custom runs dir

# Model size: "yolov8n.pt" or "yolov8s.pt"
MODEL_WEIGHTS = "yolov8n.pt"

# Image size (square). For 3024x3024, a mid-size like 1024 or 1280 is reasonable.
IMGSZ = 640

# Batch size: adjust based on 8GB VRAM. Start small and increase if OOM-free.
BATCH = 2

# Training epochs
EPOCHS = 100

# Base hyperparameters for a simple baseline
PROJECT = "yolov8_baseline"
NAME = "exp"

# --- End user config ---

def main():
    # Load a model
    model = YOLO(MODEL_WEIGHTS)

    # Train
    model.train(
        data=DATA_YAML,
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        project=PROJECT,
        name=NAME,
        device=0,
        workers=0
        # Keep baseline simple; no extra augmentation beyond defaults
        # You can later adjust these in your next experiment
        # cache=True,  # optionally cache to speed up (watch RAM)
        # workers=4,
    )

if __name__ == "__main__":
    main()


Ultralytics 8.4.14  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\First try\justtry\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=620, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [1]:
from pathlib import Path

# YOLOv8 training script with configurable augmentations
# - Update DATA_YAML to your dataset yaml path
# - Toggle augmentation groups in AUG_ENABLE to compare effects

from ultralytics import YOLO

# --- User config (edit these) ---
DATA_YAML = r"D:\First try\justtry\data.yaml"  # TODO: set your dataset yaml
RUNS_DIR = r"E:\\path\\to\\runs"               # TODO: optional; set to custom runs dir

# Model size: "yolov8n.pt" or "yolov8s.pt"
MODEL_WEIGHTS = "yolov8n.pt"

# Image size (square). For 3024x3024, a mid-size like 1024 or 1280 is reasonable.
IMGSZ = 640

# Batch size: adjust based on 8GB VRAM. Start small and increase if OOM-free.
BATCH = 2

# Training epochs
EPOCHS = 100

# Output
PROJECT = "yolov8_augmented"
NAME = "exp"

# --- Augmentation toggles ---
# Set any of these to False to disable that group.
AUG_ENABLE = {
    "color": True,        # HSV color/brightness shifts
    "rotate": True,       # free rotation for orientation invariance
    "geom": True,         # translate/scale/shear/perspective
    "flip": True,         # horizontal/vertical flips
    "mosaic": True,       # mosaic augmentation
    "mixup": True,        # mixup augmentation
    "cutmix": False,      # cutmix augmentation (optional)
    "bgr": False,         # RGB<->BGR channel swap
}

# --- Augmentation strengths (tune here) ---
# Values follow Ultralytics defaults.yaml conventions.
AUG_PARAMS = {
    # Color/lighting
    "hsv_h": 0.01,     # hue shift fraction
    "hsv_s": 0.3,     # saturation shift fraction
    "hsv_v": 0.6,      # brightness/value shift fraction (key for lighting)

    # Rotation (random in [-degrees, +degrees])
    "degrees": 180.0,  # allow any orientation

    # Geometric transforms
    "translate": 0.1,  # +/- fraction of image size
    "scale": 0.5,      # +/- scale gain
    "shear": 5.0,      # degrees
    "perspective": 0.001,  # small perspective

    # Flips
    "flipud": 0.2,     # vertical flip probability
    "fliplr": 0.5,     # horizontal flip probability

    # Compositions
    "mosaic": 0.8,     # mosaic probability
    "mixup": 0.2,      # mixup probability
    "cutmix": 0.0,     # cutmix probability

    # Channel swap
    "bgr": 0.0,        # RGB<->BGR swap probability
}

# Disable mosaic near the end to stabilize training
CLOSE_MOSAIC = 10  # epochs

# --- End user config ---


def _apply_toggle(params, key, enabled):
    if not enabled and key in params:
        params[key] = 0.0


def build_aug_params():
    params = dict(AUG_PARAMS)

    # Color group
    if not AUG_ENABLE["color"]:
        for k in ("hsv_h", "hsv_s", "hsv_v"):
            params[k] = 0.0

    # Rotation
    _apply_toggle(params, "degrees", AUG_ENABLE["rotate"])

    # Geometry
    if not AUG_ENABLE["geom"]:
        for k in ("translate", "scale", "shear", "perspective"):
            params[k] = 0.0

    # Flips
    if not AUG_ENABLE["flip"]:
        for k in ("flipud", "fliplr"):
            params[k] = 0.0

    # Mosaic / MixUp / CutMix
    _apply_toggle(params, "mosaic", AUG_ENABLE["mosaic"])
    _apply_toggle(params, "mixup", AUG_ENABLE["mixup"])
    _apply_toggle(params, "cutmix", AUG_ENABLE["cutmix"])

    # BGR
    _apply_toggle(params, "bgr", AUG_ENABLE["bgr"])

    return params


def main():
    model = YOLO(MODEL_WEIGHTS)
    aug_params = build_aug_params()

    model.train(
        data=DATA_YAML,
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        project=PROJECT,
        name=NAME,
        device=0,
        close_mosaic=CLOSE_MOSAIC,
        **aug_params,
        # cache=True,  # optionally cache to speed up (watch RAM)
        workers=2,
    )


if __name__ == "__main__":
    main()


New https://pypi.org/project/ultralytics/8.4.15 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\First try\justtry\data.yaml, degrees=180.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.6, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.8, multi_scale=0

In [5]:
from pathlib import Path

# YOLOv8 "final" training script (bigger model, long training, VRAM-friendly)
# - Prioritizes accuracy over speed
# - Uses small batch + gradient accumulation
# - Includes controlled augmentations focusing on lighting/color robustness
#
# Update DATA_YAML and (optional) RUNS_DIR before running.

from ultralytics import YOLO

# --- User config (edit these) ---
DATA_YAML = r"D:\First try\justtry\data.yaml"  # TODO: set your dataset yaml
RUNS_DIR = r"E:\\path\\to\\runs"               # TODO: optional; set to custom runs dir

# Larger model: consider "yolov8m.pt" or "yolov8l.pt" for better accuracy
MODEL_WEIGHTS = "yolov8m.pt"

# Larger input size can help small objects (watch VRAM)
IMGSZ = 1024

# Batch size + gradient accumulation to fit 8GB VRAM
BATCH = 2
ACCUMULATE = 4  # effective batch ~= BATCH * ACCUMULATE

# Long training
EPOCHS = 300

# Output
PROJECT = "yolov8_final"
NAME = "exp"

# --- Augmentation policy (balanced & controlled) ---
# Strong color/brightness jitter for lighting robustness
# Lighter geometric/composition augments to reduce over-generalization
AUG_PARAMS = {
    # Color/lighting
    "hsv_h": 0.01,
    "hsv_s": 0.2,
    "hsv_v": 0.15,

    # Rotation: allow any orientation
    "degrees": 180.0,

    # Geometric transforms (conservative)
    "translate": 0.05,
    "scale": 0.4,
    "shear": 3.0,
    "perspective": 0.0005,

    # Flips
    "flipud": 0.2,
    "fliplr": 0.5,

    # Composition (reduced vs heavy aug to avoid false positives)
    "mosaic": 0.4,
    "mixup": 0.05,
    "cutmix": 0.0,
}

# Disable mosaic near the end to stabilize
CLOSE_MOSAIC = 20

# --- Optimization ---
# Lower LR for stability in long training
LR0 = 0.002
LRF = 0.01
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 5

# Optional: label smoothing can reduce overconfidence/false positives
LABEL_SMOOTHING = 0.05

# --- End user config ---


def main():
    model = YOLO(MODEL_WEIGHTS)

    model.train(
        data=DATA_YAML,
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        #accumulate=ACCUMULATE,
        project=PROJECT,
        name=NAME,
        device=0,
        close_mosaic=CLOSE_MOSAIC,
        lr0=LR0,
        lrf=LRF,
        weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS,
        label_smoothing=LABEL_SMOOTHING,
        **AUG_PARAMS,
        # cache=True,  # optionally cache to speed up (watch RAM)
        workers=2,
    )


if __name__ == "__main__":
    main()


New https://pypi.org/project/ultralytics/8.4.15 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.14  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\First try\justtry\data.yaml, degrees=180.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.2, hsv_v=0.15, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixu